## Part 0: Setup and Imports

In [2]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import pathlib

#add model folder to path
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent.parent))

from models.cgan import create_cgan_models
from models.cgan_training import ConditionalGANTrainer, initialize_weights

# CIFAR-10 class names
CIFAR10_CLASSES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

torch.manual_seed(42)
np.random.seed(42)

# Determine device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: cpu


In [3]:
# Visualization helpers 
def plot_loss_curves_cgan(results):
    """Plot discriminator and generator losses for cGAN."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    d_losses = np.array(results["d_losses"])
    g_losses = np.array(results["g_losses"])

    # Plot D loss
    axes[0].plot(d_losses, linewidth=2, color="navy", label="D Loss")
    axes[0].axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Ideal (0.5)")
    axes[0].set_xlabel("Training Step", fontsize=11)
    axes[0].set_ylabel("BCE Loss", fontsize=11)
    axes[0].set_title(
        "Discriminator Loss Over Training", fontsize=12, fontweight="bold"
    )
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    # Plot G loss
    axes[1].plot(g_losses, linewidth=2, color="darkgreen", label="G Loss")
    axes[1].set_xlabel("Training Step", fontsize=11)
    axes[1].set_ylabel("BCE Loss", fontsize=11)
    axes[1].set_title("Generator Loss Over Training", fontsize=12, fontweight="bold")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Print statistics
    print(f"D Loss - Mean: {d_losses.mean():.4f}, Std: {d_losses.std():.4f}")
    print(f"G Loss - Mean: {g_losses.mean():.4f}, Std: {g_losses.std():.4f}")


def visualize_class_grid(grid_images, class_names, title="cGAN Class Disentanglement"):
    """Visualize 10x10 class disentanglement grid."""
    grid_images_cpu = (grid_images.cpu() + 1) / 2
    grid_images_cpu = torch.clamp(grid_images_cpu, 0, 1)

    fig, axes = plt.subplots(10, 10, figsize=(16, 16))

    for class_id in range(10):
        for sample_id in range(10):
            idx = class_id * 10 + sample_id
            ax = axes[class_id, sample_id]
            img = grid_images_cpu[idx].cpu().permute(1, 2, 0).numpy()
            ax.imshow(img)
            ax.axis("off")
            if sample_id == 0:
                ax.set_ylabel(class_names[class_id], fontsize=10, fontweight="bold")

    plt.suptitle(
        f"{title}\n(Rows: Classes | Columns: Same Noise, Different Classes)",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


def visualize_class_samples(class_samples, class_name, title="Generated Samples"):
    """Visualize 16 samples from a specific class."""
    class_samples_cpu = (class_samples.cpu() + 1) / 2
    class_samples_cpu = torch.clamp(class_samples_cpu, 0, 1)

    fig, axes = plt.subplots(2, 8, figsize=(16, 4))

    for i in range(16):
        ax = axes[i // 8, i % 8]
        img = class_samples_cpu[i].cpu().permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.axis("off")

    plt.suptitle(
        f'16 Generated Samples from Class "{class_name}"',
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


print(" Visualization helpers loaded")


 Visualization helpers loaded


---

## Part 1: Load CIFAR-10 Dataset

**TODO:** Load the CIFAR-10 dataset with appropriate transforms.

**Requirements:**
- Normalize images to [-1, 1] range (using mean=0.5, std=0.5)
- Create DataLoader with batch_size=64
- Enable shuffling for training

In [4]:
# TODO 1: Load CIFAR-10 dataset with transforms
# Step 1a: Define transforms (ToTensor + Normalize to [-1, 1])
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

# Step 1b: Load training dataset
train_dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

# Step 1c: Create DataLoader
batch_size = 64
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=0  # MPS compatibility
)

print(f"Dataset loaded: {len(train_dataset)} training images")
print(f"DataLoader: {len(train_loader)} batches of size {batch_size}")


100%|██████████| 170M/170M [44:50<00:00, 63.4kB/s]   


Dataset loaded: 50000 training images
DataLoader: 782 batches of size 64


---

## Part 2: Create cGAN Models

**TODO:** Create the conditional generator and discriminator.

**Requirements:**
- Use `create_cgan_models()` function
- Set latent_dim=100 (noise vector dimension)
- Set num_classes=10 (CIFAR-10 has 10 classes)
- Initialize weights using DCGAN guidelines

In [5]:
# TODO 2: Create cGAN models
latent_dim = 100
num_classes = 10
label_dim = 50  # Label embedding dimension

# Step 2a: Create generator and discriminator
generator, discriminator = create_cgan_models(
    latent_dim=latent_dim,
    num_classes=num_classes,
    label_dim=label_dim,
    num_channels=3,
    device=device
)

# Step 2b: Initialize weights
initialize_weights(generator)
initialize_weights(discriminator)

# Verify model parameters
print(f"Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
print(
    f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}"
)


Generator parameters: 2,869,172
Discriminator parameters: 1,373,109


---

## Part 3: Create Trainer

**TODO:** Instantiate the ConditionalGANTrainer.

**Requirements:**
- Pass generator and discriminator
- Set lr_g=0.0002 (Generator learning rate)
- Set lr_d=0.0002 (Discriminator learning rate)
- Use Adam optimizer with beta1=0.5, beta2=0.999

In [6]:
# TODO 3: Create trainer
trainer = ConditionalGANTrainer(
    generator=generator,
    discriminator=discriminator,
    device=device,
    lr_g=0.0002,
    lr_d=0.0002,
    beta1=0.5,
    beta2=0.999,
)

print("Trainer initialized successfully")
print(f"Learning rates - G: 0.0002, D: 0.0002")


Trainer initialized successfully
Learning rates - G: 0.0002, D: 0.0002


---

## Part 4: Train the cGAN

**TODO:** Train the model for 20 epochs.

**Requirements:**
- Use trainer.train() method
- Set num_epochs=20
- Pass latent_dim and num_classes
- Log progress every 50 batches

**Note:** This will take 20-30 minutes on GPU.

In [7]:
# TODO 4: Train the cGAN
num_epochs = 20

print(f"Training cGAN for {num_epochs} epochs...")
print("This will take 20-30 minutes on GPU\n")

results = trainer.train(
    train_loader=train_loader,
    num_epochs=num_epochs,
    latent_dim=latent_dim,
    num_classes=num_classes,
    log_interval=50,
)

print(f"\n Training complete after {num_epochs} epochs")


Training cGAN for 20 epochs...
This will take 20-30 minutes on GPU


Starting Conditional GAN (cGAN) Training
Device: cpu
Epochs: 20
Classes: 10
Dataset size: 50048

Epoch [1/20] Batch [50/782] D_loss: 1.1485 | G_loss: 1.8803
Epoch [1/20] Batch [100/782] D_loss: 2.5205 | G_loss: 2.7115
Epoch [1/20] Batch [150/782] D_loss: 0.2957 | G_loss: 3.7923
Epoch [1/20] Batch [200/782] D_loss: 0.9043 | G_loss: 1.6262
Epoch [1/20] Batch [250/782] D_loss: 0.7095 | G_loss: 2.2850
Epoch [1/20] Batch [300/782] D_loss: 0.4849 | G_loss: 2.2199
Epoch [1/20] Batch [350/782] D_loss: 1.0269 | G_loss: 1.8560
Epoch [1/20] Batch [400/782] D_loss: 1.1234 | G_loss: 2.1563
Epoch [1/20] Batch [450/782] D_loss: 0.7027 | G_loss: 1.9950
Epoch [1/20] Batch [500/782] D_loss: 0.6661 | G_loss: 3.0355
Epoch [1/20] Batch [550/782] D_loss: 0.9009 | G_loss: 1.9094
Epoch [1/20] Batch [600/782] D_loss: 1.0053 | G_loss: 1.9891
Epoch [1/20] Batch [650/782] D_loss: 0.8193 | G_loss: 2.1181
Epoch [1/20] Batch [700/782] D_loss: 0.884

---

## Part 5: Plot Loss Curves

**TODO:** Visualize Generator and Discriminator losses during training.

**Requirements:**
- Create 1×2 subplot (D loss, G loss)
- Plot results['d_losses'] and results['g_losses']
- Add labels and titles
- Add reference line at y=0.5 (ideal discriminator loss)

In [ ]:
# TODO 5: Plot loss curves
plot_loss_curves_cgan(results)

---

## Part 6: Generate 10×10 Class Disentanglement Grid

**TODO:** Generate a 10×10 grid showing all classes with the same noise vector.

**Requirements:**
- Use trainer.generate_all_classes_grid()
- Generate 10 classes × 10 samples per class = 100 images
- Same noise z, different class labels y
- Denormalize from [-1,1] to [0,1]
- Visualize as 10×10 subplot grid

In [ ]:
# TODO 6: Generate 10x10 class disentanglement grid
print("Generating 10×10 class disentanglement grid...")

# Step 6a: Generate grid
grid_images = trainer.generate_all_classes_grid(
    num_classes=10,
    samples_per_class=10,
    latent_dim=latent_dim,
    shared_z=None,
)

# Step 6b: Visualize using helper
visualize_class_grid(
    grid_images, CIFAR10_CLASSES, title="cGAN Class Disentanglement: 10×10 Grid"
)

print(f" Grid generated: {grid_images.shape}")


---

## Part 7: Generate Single-Class Samples

**TODO:** Generate 16 images of a specific class (e.g., dogs).

**Requirements:**
- Use trainer.generate_class_samples()
- Set target_class=5 (dogs)
- Generate num_samples=16
- Denormalize to [0,1]
- Display as 2×8 grid

In [ ]:
# TODO 7: Generate single-class samples
target_class = 5  # dogs

print(f"Generating 16 samples of class '{CIFAR10_CLASSES[target_class]}'...")

# Step 7a: Generate samples
class_samples = trainer.generate_class_samples(
    target_class=target_class,
    num_samples=16,
    latent_dim=latent_dim
)

# Step 7b: Visualize using helper
visualize_class_samples(class_samples, CIFAR10_CLASSES[target_class])

print(
    f"✓ Generated 16 samples from class {CIFAR10_CLASSES[target_class]} (class {target_class})"
)


---

## Summary

### What You've Learned

 **Conditional GANs:** How to add class information to Generator and Discriminator

 **Label Conditioning:** Early concatenation (G) vs late concatenation (D)


### Next Steps to Improve

+ Train for more epochs (50+) for better quality
+ Use generated images to train a classifier and measure improvement
+ Experiment with CIFAR-100 or ImageNet subsets
+ Compare with other conditional architectures